# Introduction to Single-Cell Perturbation Analysis


Single-cell perturbation analysis is a framework for identifying how a biological treatment, stimulus, or genetic perturbation affects cellular states at the single-cell level. Rather than comparing bulk averages between treated and control samples, this approach leverages single-cell resolution to quantify subtle and cell-type–specific responses to a perturbation.

Perturbation analysis in single-cell data often revolves around the idea of separability: if a perturbation, could be checical, biological or genetic (CRISPRi) has a strong effect on a given cell population, it should be easy to distinguish treated from untreated cells based on their transcriptomes. Conversely, if the perturbation barely affects a cell type, classification between treated and untreated will be difficult.

In our case, we have two single-cell RNA-seq datasets: one representing cells exposed to interferon beta (IFN-β) after 6 hours and another representing unexposed control cells. The goal is to determine which cell types are most responsive to interferon signaling and how strongly their transcriptional profiles shift in response.


### Installations

In [ ]:
!pip install scanpy
!pip install anndata
!pip3 install igraph
!pip install celltypist
!pip install decoupler
!pip install fa2-modified
!pip install louvain
!pip install pertpy

In [ ]:
!pip install harmonypy

### Import Libraries

In [ ]:
#Import core single cell datasets

import scanpy as sc
import anndata as ad
import numpy as np
import pertpy as pt
import pandas as pd


### Download the cells without IFN stimulation

In [ ]:
!wget https://github.com/josoga2/sc/raw/refs/heads/main/GSE96583_RAW/GSM2560249_2.2.mtx.gz
!wget https://github.com/josoga2/sc/raw/refs/heads/main/GSE96583_RAW/GSM2560249_barcodes.tsv.gz
!wget https://github.com/josoga2/sc/raw/refs/heads/main/GSE96583_RAW/GSE96583_batch2.genes.tsv.gz

In [ ]:
!mkdir -p GSM2560245_Before
!mv GSM2560249_2.2.mtx.gz /content/GSM2560245_Before
!mv GSM2560249_barcodes.tsv.gz /content/GSM2560245_Before
!mv GSE96583_batch2.genes.tsv.gz /content/GSM2560245_Before

#rename files to matrix, barcode and genes
!mv /content/GSM2560245_Before/GSM2560249_2.2.mtx.gz /content/GSM2560245_Before/matrix.mtx.gz
!mv /content/GSM2560245_Before/GSM2560249_barcodes.tsv.gz /content/GSM2560245_Before/barcodes.tsv.gz
!mv /content/GSM2560245_Before/GSE96583_batch2.genes.tsv.gz /content/GSM2560245_Before/genes.tsv.gz

### Download the cells with IFN stimulation after 6 hrs

In [ ]:
!wget https://github.com/josoga2/sc/raw/refs/heads/main/GSE96583_RAW/GSM2560248_2.1.mtx.gz
!wget https://github.com/josoga2/sc/raw/refs/heads/main/GSE96583_RAW/GSM2560248_barcodes.tsv.gz
!wget https://github.com/josoga2/sc/raw/refs/heads/main/GSE96583_RAW/GSE96583_batch2.genes.tsv.gz

In [ ]:
!mkdir -p GSM2560245_After
!mv GSM2560248_2.1.mtx.gz /content/GSM2560245_After
!mv GSM2560248_barcodes.tsv.gz /content/GSM2560245_After
!mv GSE96583_batch2.genes.tsv.gz /content/GSM2560245_After

#rename the files accordingly
!mv /content/GSM2560245_After/GSM2560248_2.1.mtx.gz /content/GSM2560245_After/matrix.mtx.gz
!mv /content/GSM2560245_After/GSM2560248_barcodes.tsv.gz /content/GSM2560245_After/barcodes.tsv.gz
!mv /content/GSM2560245_After/GSE96583_batch2.genes.tsv.gz /content/GSM2560245_After/genes.tsv.gz

In [ ]:
#write a function for assembling the dataset

def load_one_dataset(path, condition_label):
    # Read matrix
    adata = sc.read_mtx(f"{path}/matrix.mtx.gz").T

    # Read barcodes
    adata.obs_names = pd.read_csv(f"{path}/barcodes.tsv.gz", header=None, sep="\t")[0]

    # Read and clean genes
    genes = pd.read_csv(f"{path}/genes.tsv.gz", header=None, sep = '\t')[1]
    #print(genes.head())
    genes = genes.astype(str).str.replace(r'^\d+"', '', regex=True)
    adata.var_names = genes.var_names = genes[:adata.shape[1]]

    #make unique
    adata.var_names_make_unique()
    adata.obs_names_make_unique()

    # Add condition label
    adata.obs["condition"] = condition_label

    return adata

In [ ]:
adata_before = load_one_dataset("/content/GSM2560245_After", "before_IFN_beta")
adata_after = load_one_dataset("/content/GSM2560245_Before", "after_IFN_beta")

In [ ]:
ifn_adata = adata_before.concatenate(adata_after, batch_key="condition",
                                 batch_categories=["before_IFN_beta", "after_IFN_beta"])

In [ ]:
ifn_adata.var.head()

### The Core Pipeline!

In [ ]:
ifn_adata.var['MT'] = ifn_adata.var_names.str.startswith("MT-")
ifn_adata.var['RIBO'] = ifn_adata.var_names.str.startswith("RPS", "RPL")
ifn_adata.var['HB'] = ifn_adata.var_names.str.startswith("^HB[^(P)]")

In [ ]:
sc.pp.calculate_qc_metrics(
    ifn_adata, qc_vars=["MT", 'RIBO', 'HB'], inplace=True, log1p=True
)

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (5,4)  # Adjust figure size
plt.rcParams["axes.grid"] = True  # Add grid to plots
plt.rcParams["axes.edgecolor"] = "black" # Set plot border color
plt.rcParams["axes.linewidth"] = 1.5 # Set plot border width
plt.rcParams["axes.facecolor"] = "white" # Set background color
plt.rcParams["axes.labelcolor"] = "black" # Set label color
plt.rcParams["xtick.color"] = "black" # Set x-axis tick color
plt.rcParams["ytick.color"] = "black" # Set y-axis tick color
plt.rcParams["text.color"] = "black" # Set text color
%matplotlib inline

In [ ]:
sc.pl.violin(
    ifn_adata,
    ["n_genes_by_counts", 'total_counts', 'pct_counts_MT'],
    jitter=0.4,
    multi_panel=False,
)



In [ ]:
ifn_adata.obs_keys()

In [ ]:
sc.pl.scatter(ifn_adata, "total_counts", "n_genes_by_counts")

In [ ]:
#Dim Reduction
sc.tl.pca(ifn_adata)
sc.pl.pca_variance_ratio(ifn_adata, n_pcs=10, log=False)

In [ ]:
#Let's correct for batch effect
import scanpy.external as sce
sce.pp.harmony_integrate(ifn_adata, key="condition")

In [ ]:
#Normalisation
ifn_adata.layers["counts"] = ifn_adata.X.copy()
sc.pp.normalize_total(ifn_adata)
sc.pp.log1p(ifn_adata)

In [ ]:
#Feature selection
sc.pp.highly_variable_genes(ifn_adata, n_top_genes=2500)
sc.pl.highly_variable_genes(ifn_adata)

In [ ]:
#Dim Reduction
sc.tl.pca(ifn_adata)
sc.pl.pca_variance_ratio(ifn_adata, n_pcs=10, log=False)


In [ ]:
ifn_adata.var_names

In [ ]:
sc.pl.pca(ifn_adata, cmap="coolwarm", color = ['total_counts_RIBO', 'condition'])

In [ ]:
sc.pp.neighbors(ifn_adata)
sc.tl.umap(ifn_adata)

In [ ]:
sc.pl.umap(
    ifn_adata,
    size=8,
    color=["condition"],
)

In [ ]:
sc.tl.leiden(ifn_adata, flavor="igraph", n_iterations=10, key_added="leiden_res_", resolution=0.5 )

In [ ]:
sc.pl.umap(
    ifn_adata,
    color=["leiden_res_"],
    size=8,
)

In [ ]:
import decoupler as dc

In [ ]:
#!wget wget -O result.txt 'http://www.ensembl.org/biomart/martservice?query=<?xml version="1.0" encoding="UTF-8"?><!DOCTYPE Query><Query  virtualSchemaName = "default" formatter = "CSV" header = "0" uniqueRows = "0" count = "" datasetConfigVersion = "0.6" ><Dataset name = "hsapiens_gene_ensembl" interface = "default" ><Attribute name = "ensembl_gene_id" /><Attribute name = "external_gene_name" /></Dataset></Query>'

In [ ]:
#import pandas as pd
#ensembl_var = pd.read_csv('/content/result.txt', header = None)
#ensembl_var.columns = ['ensembl_gene_id', 'gene_name']
#ensembl_var.head(3)

In [ ]:
# Query Omnipath and get PanglaoDB
markers = dc.op.resource(name="PanglaoDB", organism="human")

# Remove duplicated entries
markers = markers[~markers.duplicated(["cell_type", "genesymbol"])]

#Format because dc only accepts cell_type and genesymbol

markers = markers.rename(columns={"cell_type": "source", "genesymbol": "target"})
markers = markers[["source", "target"]]


markers.head()

In [ ]:
ifn_adata.var_names

In [ ]:
dc.mt.ulm(data=ifn_adata,
          net=markers,
          tmin = 3)

In [ ]:
score = dc.pp.get_obsm(ifn_adata, key="score_ulm")

In [ ]:
ifn_adata.obsm["score_ulm"].head(1)

In [ ]:
ifn_adata.obsm["score_ulm"].columns

In [ ]:
#rank genes
ifn_gene_rank = dc.tl.rankby_group(score, groupby="leiden_res_", reference="rest", method="t-test_overestim_var")
ifn_gene_rank = ifn_gene_rank[ifn_gene_rank["stat"] > 0]
ifn_gene_rank.head(5)

In [ ]:
top_names_per_group = ifn_gene_rank.groupby('group')['name'].apply(lambda x: x.head(1))
display(top_names_per_group)

In [ ]:
sc.pl.umap(score, color=["Monocytes", 'leiden_res_'], cmap="RdBu_r")

In [ ]:
n_ctypes = 1
ctypes_dict = ifn_gene_rank.groupby("group").head(n_ctypes).groupby("group")["name"].apply(lambda x: list(x)).to_dict()
ctypes_dict

In [ ]:
dict_ann = ifn_gene_rank[ifn_gene_rank["stat"] > 0].groupby("group").head(1).set_index("group")["name"].to_dict()
dict_ann

In [ ]:
dict_ann_unique = {k: v + '_' + str(k) for k, v in dict_ann.items()}
display(dict_ann_unique)

In [ ]:
ifn_adata.obs["leiden_res_"] = ifn_adata.obs["leiden_res_"].cat.rename_categories(dict_ann_unique)

In [ ]:

plt.rcParams["figure.figsize"] = (5,4)  # Adjust figure size
# Reduce the default font size for text elements in plots
plt.rcParams['font.size'] = 8
plt.rcParams['axes.labelsize'] = 8
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8

# Now replot the UMAP
sc.pl.umap(
    adata=ifn_adata,
    color=[ "leiden_res_"],
    ncols=2,
    legend_loc = 'on data',
    size=4,
)

### Perturbation modelling

How it works:
- **Input Data**: A single-cell expression matrix with cell-level metadata indicating the experimental condition (e.g., `condition = ['control', 'IFN_beta']`) and cell type annotations.
- **Subsampling**: To avoid biases due to unequal cell numbers, Augur repeatedly subsamples a fixed number of cells per condition per cell type.
- **Feature Selection**: It selects variable genes to focus on biologically relevant variation.
- **Classification**: For each cell type, Augur trains a machine learning model—typically a random forest—to predict which condition each cell belongs to.
- **Performance Scoring**: The mean AUC across subsampling iterations is computed. A high AUC indicates that the perturbation strongly shifts that cell type’s expression profile, making it easier to classify. A low AUC suggests minimal transcriptional change.



Here, we will use our dataset to know which cells are best targeted during stimulation IFN-β stimulation

In [ ]:
ifn_adata.layers.keys()

In [ ]:
raw_counts = ifn_adata.layers["counts"]

In [ ]:
raw_counts.shape

In [ ]:
raw_counts

In [ ]:
ifn_adata.X = raw_counts

In [ ]:
ifn_adata.obs_keys()

In [ ]:
ifn_adata.obs["condition"]

In [ ]:
ifn_adata.obs.leiden_res_.value_counts()

In [ ]:
ag_rfc = pt.tl.Augur("random_forest_classifier")

In [ ]:
loaded_data = ag_rfc.load(ifn_adata, label_col="condition", cell_type_col="leiden_res_")
loaded_data

In [ ]:
v_adata, v_results = ag_rfc.predict(
    loaded_data, subsample_size=50, n_threads=4, select_variance_features=True, span=1
)

v_results["summary_metrics"]

In [ ]:
lollipop = ag_rfc.plot_lollipop(v_results)

In [ ]:
important_features = ag_rfc.plot_important_features(v_results)